# 2D FEP Subtraction — Glycine Calibration Trial
### SSA — Diamond I11 Beamline | Run7_GLY_0.5VF_X2

**Purpose:** Calibrate the frame classifier using glycine data where 50 crystal hits
are already known from visual inspection. Glycine was strongly diffracting so the
Bragg spot signal above FEP background should be clearly separable from non-crystal frames.

**Ground truth available:** 50 confirmed crystal files provided — these are used to
find the natural score threshold that separates hits from misses.

---
**Run cells one at a time. Steps 0–3 are setup. Step 4 is the key calibration step.**

---
## Step 0 — Paths and ground truth

In [ ]:
import os, glob, numpy as np

# ══════════════════════════════════════════════════════════════
# PATHS — confirm these before running
# ══════════════════════════════════════════════════════════════

RAW_DATA_DIR = r"E:/I11BT_dec25_dlm_gly/Data_Processing/RAW_2D/Run7_GLY_0.5VF_X2/"
FILE_PATTERN = "i11-1-*.nxs"          # primary pattern
FILE_PATTERN_HDF = "pixium_*.hdf"     # fallback if .nxs not present
OUTPUT_DIR   = r"E:/I11BT_dec25_dlm_gly/Data_Processing/Trial_2D_Corrected/Glycine/"
MASK_PATH    = r"E:/I11BT_dec25_dlm_gly/Data_Processing/X2_Calib/X2_mask.npy"
PONI_PATH    = r"E:/I11BT_dec25_dlm_gly/Data_Processing/X2_Calib/X2_calib.poni"

# ══════════════════════════════════════════════════════════════
# GROUND TRUTH — 50 confirmed crystal collection numbers
# from visual inspection of pixium_NNNNNN.hdf files
# ══════════════════════════════════════════════════════════════
KNOWN_CRYSTAL_NUMBERS = [
    "122988","122993","122995","122996","122998","122999","123002","123012",
    "123022","123027","123037","123040","123042","123043","123056","123061",
    "123062","123069","123070","123071","123079","123081","123083","123085",
    "123086","123090","123093","123099","123100","123101","123102","123104",
    "123107","123108","123109","123110","123111","123112","123121","123122",
    "123131","123141","123142","123146","123148","123149","123150","123155",
    "123162","123172"
]

# ── Resolve file list (try .nxs first, fall back to .hdf) ──────────────────
all_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, FILE_PATTERN)))
if len(all_files) == 0:
    all_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, FILE_PATTERN_HDF)))
    print(f"Using .hdf files: {len(all_files)} found")
else:
    print(f"Using .nxs files: {len(all_files)} found")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load mask ──────────────────────────────────────────────────────────────
mask = np.load(MASK_PATH)
print(f"Mask loaded: {mask.shape}  masked pixels: {mask.sum():,} ({100*mask.sum()/mask.size:.1f}%)")

# ── Identify ground truth files ────────────────────────────────────────────
def extract_number(fp):
    stem = os.path.splitext(os.path.basename(fp))[0]
    for part in reversed(stem.replace("-","_").split("_")):
        if part.isdigit():
            return part
    return ""

all_numbers   = [extract_number(f) for f in all_files]
crystal_files = [f for f, n in zip(all_files, all_numbers) if n in KNOWN_CRYSTAL_NUMBERS]
miss_files    = [f for f, n in zip(all_files, all_numbers) if n not in KNOWN_CRYSTAL_NUMBERS]

print(f"\nTotal files   : {len(all_files)}")
print(f"Crystal hits  : {len(crystal_files)}  (from ground truth list)")
print(f"Misses        : {len(miss_files)}")

if len(crystal_files) == 0:
    print("\n⚠️  No ground truth files matched. Check FILE_PATTERN and RAW_DATA_DIR.")
    print("   The collection number must appear in the filename.")
    print("   First 5 filenames found:")
    for f in all_files[:5]:
        print(f"     {os.path.basename(f)}  ->  extracted number: '{extract_number(f)}'")
else:
    print(f"\n✅ Ground truth matched. Continue to Step 1.")

---
## Step 1 — Import module

In [ ]:
from fep_subtraction_2d_i11 import (
    compute_frame_metrics,
    _load_pixium_frame,
    _read_exposure_time,
    classify_dataset,
    build_background_from_dataset,
    process_dataset,
)
print("✅ Module imported.")

---
## Step 2 — Inspect file structure

Confirms detector path and frame shape for the glycine .hdf files.
The path may differ from the DLM .nxs files.

In [ ]:
from fep_subtraction_2d_i11 import inspect_nexus
import logging
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')

inspect_nexus(all_files[0])

frame = _load_pixium_frame(all_files[0])
print(f"Frame shape : {frame.shape}")
print(f"dtype       : {frame.dtype}")
print(f"min/max     : {frame.min():.0f} / {frame.max():.0f}")
print(f"mean        : {frame.mean():.1f}")

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

---
## Step 3 — Confirm FEP ring position on glycine data

The classifier uses r=255px as the FEP ring centre (confirmed on DLM data).
This step verifies that position is consistent on the glycine .hdf files,
which may have been collected with slightly different detector distance.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, fp, label in zip(axes,
    [crystal_files[0], miss_files[0]],
    ["Known crystal hit", "Known miss"]):
    
    frame = _load_pixium_frame(fp)
    if frame.ndim == 3:
        frame = frame[0]
    arr = frame.astype(np.float64)
    arr_m = arr.copy()
    arr_m[mask.astype(bool)] = 0.0
    
    rows, cols = arr.shape
    cy, cx = rows/2.0, cols/2.0
    y, x = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x-cx)**2 + (y-cy)**2).astype(int)
    r_max = r_map.max()
    s = np.bincount(r_map.ravel(), weights=arr_m.ravel(), minlength=r_max+1)
    c = np.bincount(r_map.ravel(), minlength=r_max+1)
    with np.errstate(invalid='ignore', divide='ignore'):
        radial = np.where(c>0, s/c, 0.0)
    
    ax.plot(radial[:800], lw=1)
    fep_r = int(np.argmax(radial[80:600]) + 80)
    ax.axvline(fep_r, color='red', ls='--', lw=1, label=f'FEP peak r={fep_r}px')
    ax.set_xlabel("Radius (pixels)")
    ax.set_ylabel("Mean intensity")
    ax.set_title(f"{label}\n{os.path.basename(fp)}")
    ax.legend()
    ax.grid(alpha=0.3)
    print(f"{label}: FEP peak at r={fep_r}px")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "glycine_radial_profiles.png"), dpi=150)
plt.show()
print("\nIf both red dashed lines are near r=255px the classifier geometry is correct.")
print("If they differ significantly, note the value — we will update FEP_RING_R below.")

---
## Step 4 — Threshold calibration using ground truth ← KEY STEP

This is the core calibration. We compute `spot_density` (the primary
classification metric) for every frame using the known crystal/miss labels,
then find the natural gap in the score distribution.

The classifier currently uses `CRYSTAL_THRESHOLD = 0.15`. This step tells
us whether that threshold is correct for your detector, or what it should be.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# ── Parameters — update FEP_RING_R if Step 3 showed a different value ──
FEP_RING_R = 255    # pixels — main FEP ring radius (update from Step 3 if needed)
BEAM_EXCL  = 60     # pixels — exclude direct beam / beamstop core
OUTER_EXCL = 1200   # pixels — exclude outer detector edge

def compute_spot_density(filepath, mask, fep_r=FEP_RING_R):
    """
    Core metric: fraction of valid pixels with residual > N×std above
    the smooth radial background, outside the direct beam region.
    Returns a dict of spot_density at different std thresholds.
    """
    frame = _load_pixium_frame(filepath)
    if frame.ndim == 3:
        frame = frame[0]
    arr = frame.astype(np.float64)
    arr[mask.astype(bool)] = 0.0

    rows, cols = arr.shape
    cy, cx = rows/2.0, cols/2.0
    y, x = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x-cx)**2 + (y-cy)**2).astype(np.int32)
    r_max = int(r_map.max())

    s = np.bincount(r_map.ravel(), weights=arr.ravel(), minlength=r_max+1)
    c = np.bincount(r_map.ravel(), minlength=r_max+1)
    with np.errstate(invalid='ignore', divide='ignore'):
        radial_mean = np.where(c>0, s/c, 0.0)

    residual = arr - radial_mean[r_map]
    valid = (r_map >= BEAM_EXCL) & (r_map <= OUTER_EXCL) & ~mask.astype(bool)
    res_v = residual[valid]
    std   = res_v.std()
    n     = valid.sum()

    return {
        "2std": float(np.sum(res_v > 2*std) / n),
        "3std": float(np.sum(res_v > 3*std) / n),
        "5std": float(np.sum(res_v > 5*std) / n),
        "8std": float(np.sum(res_v > 8*std) / n),
        "10std": float(np.sum(res_v > 10*std) / n),
        "std":  float(std),
        "mean": float(res_v.mean()),
    }

print("Computing spot_density for all files...")
print("(This may take a few minutes for 296 files)\n")

crystal_scores = []
miss_scores    = []

# Sample up to 80 miss files to keep runtime reasonable
import random
miss_sample = miss_files[:80] if len(miss_files) > 80 else miss_files

for i, fp in enumerate(crystal_files):
    d = compute_spot_density(fp, mask)
    crystal_scores.append(d)
    if (i+1) % 10 == 0:
        print(f"  Crystal: {i+1}/{len(crystal_files)}")

for i, fp in enumerate(miss_sample):
    d = compute_spot_density(fp, mask)
    miss_scores.append(d)
    if (i+1) % 20 == 0:
        print(f"  Miss:    {i+1}/{len(miss_sample)}")

print("\nDone.")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# ── Show the distribution for each threshold ──────────────────────────────
thresholds = ["3std", "5std", "8std", "10std"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

best_threshold = None
best_separation = 0

for ax, thr in zip(axes, thresholds):
    c_vals = np.array([s[thr] for s in crystal_scores]) * 100  # as %
    m_vals = np.array([s[thr] for s in miss_scores])   * 100
    
    # Find natural gap
    all_vals = np.concatenate([c_vals, m_vals])
    vmin, vmax = all_vals.min(), all_vals.max()
    bins = np.linspace(vmin, vmax, 40)
    
    ax.hist(c_vals, bins=bins, alpha=0.6, color='teal',  label=f'Crystal hits (n={len(c_vals)})')
    ax.hist(m_vals, bins=bins, alpha=0.6, color='coral', label=f'Misses (n={len(m_vals)})')
    ax.set_xlabel("Spot density (%)")
    ax.set_ylabel("Frame count")
    ax.set_title(f"Threshold: residual > {thr}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    # Measure separation (difference in means relative to spread)
    sep = abs(c_vals.mean() - m_vals.mean()) / (c_vals.std() + m_vals.std() + 1e-9)
    ax.set_xlabel(f"Spot density (%)  |  separation={sep:.2f}")
    
    if sep > best_separation:
        best_separation = sep
        best_threshold = thr

plt.suptitle("Spot density distributions: crystal hits vs misses\n"
             "(well-separated = good threshold for classification)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "threshold_calibration.png"), dpi=150)
plt.show()

print(f"Best separating threshold: {best_threshold} (separation score: {best_separation:.2f})")
print()
print("=== Summary statistics ===")
for thr in thresholds:
    c_vals = np.array([s[thr] for s in crystal_scores]) * 100
    m_vals = np.array([s[thr] for s in miss_scores])   * 100
    print(f"\n  {thr}:")
    print(f"    Crystal:  mean={c_vals.mean():.4f}%  std={c_vals.std():.4f}%  "
          f"min={c_vals.min():.4f}%  max={c_vals.max():.4f}%")
    print(f"    Miss:     mean={m_vals.mean():.4f}%  std={m_vals.std():.4f}%  "
          f"min={m_vals.min():.4f}%  max={m_vals.max():.4f}%")
    
    # Suggest threshold
    crystal_min = c_vals.min()
    miss_max    = m_vals.max()
    if crystal_min > miss_max:
        midpoint = (crystal_min + miss_max) / 2
        print(f"    ✅ CLEAN SEPARATION — suggested threshold: {midpoint:.4f}%")
    else:
        overlap = np.sum(m_vals >= crystal_min)
        print(f"    ⚠️  {overlap} miss frames overlap with crystal minimum")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# ── Residual images: crystal hit vs miss side by side ─────────────────────
# Visual confirmation that the metric is capturing Bragg spots

def make_residual(filepath, mask):
    frame = _load_pixium_frame(filepath)
    if frame.ndim == 3:
        frame = frame[0]
    arr = frame.astype(np.float64)
    arr[mask.astype(bool)] = 0.0
    rows, cols = arr.shape
    cy, cx = rows/2.0, cols/2.0
    y, x = np.ogrid[:rows, :cols]
    r_map = np.sqrt((x-cx)**2 + (y-cy)**2).astype(np.int32)
    r_max = int(r_map.max())
    s = np.bincount(r_map.ravel(), weights=arr.ravel(), minlength=r_max+1)
    c = np.bincount(r_map.ravel(), minlength=r_max+1)
    with np.errstate(invalid='ignore', divide='ignore'):
        rm = np.where(c>0, s/c, 0.0)
    return frame, arr - rm[r_map]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col, (fp, label) in enumerate([
    (crystal_files[0], f"Crystal hit\n{os.path.basename(crystal_files[0])}"),
    (miss_files[0],    f"Miss\n{os.path.basename(miss_files[0])}")
]):
    raw, residual = make_residual(fp, mask)
    if raw.ndim == 3: raw = raw[0]
    
    from matplotlib.colors import LogNorm
    vmin = max(1, np.percentile(raw[raw>0], 1))
    vmax = np.percentile(raw, 99.5)
    axes[0, col].imshow(raw, origin='lower',
                         norm=LogNorm(vmin=vmin, vmax=vmax),
                         cmap='viridis', aspect='auto')
    axes[0, col].set_title(f"Raw frame\n{label}", fontsize=9)
    
    # Residual — clip to ±5σ for display
    valid = ~mask.astype(bool)
    std = residual[valid].std()
    axes[1, col].imshow(residual, origin='lower',
                         vmin=-3*std, vmax=5*std,
                         cmap='RdBu_r', aspect='auto')
    axes[1, col].set_title(
        f"Residual after radial subtraction\n"
        f"Red = above ring average (Bragg spots appear here)", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "residual_images_crystal_vs_miss.png"), dpi=150)
plt.show()
print("\nThe crystal hit residual should show bright red spots scattered across the frame.")
print("The miss residual should look like uniform noise with no obvious bright spots.")

---
## Step 5 — Set thresholds and rerun classification

Based on the distributions in Step 4, set the thresholds below and rerun
`classify_dataset()`. The goal is: ~50 crystal, ~246 non-crystal.

**If Step 4 showed clean separation**, the suggested midpoint is your threshold.
**If Step 4 showed overlap**, set the threshold conservatively high (fewer false
crystal positives is better than contaminating the background pool).

In [ ]:
# ══════════════════════════════════════════════════════════════
# SET THESE BASED ON THE STEP 4 OUTPUT
# ══════════════════════════════════════════════════════════════

# Which std multiplier gave the best separation? (from Step 4)
# Options: "3std", "5std", "8std", "10std"
BEST_STD_MULTIPLIER = 5      # e.g. if "5std" was best, put 5 here

# What spot_density threshold separates crystal from miss?
# Use the suggested midpoint from Step 4 output (as a fraction, not %)
# e.g. if midpoint was 0.0200%, enter 0.0002
CRYSTAL_THRESHOLD_PCT  = 0.020   # % — above this = crystal
SOLUTION_THRESHOLD_PCT = 0.005   # % — below this = solution/gas

print(f"Thresholds to apply:")
print(f"  Std multiplier : {BEST_STD_MULTIPLIER}×")
print(f"  Crystal  >     : {CRYSTAL_THRESHOLD_PCT:.4f}%")
print(f"  Solution <     : {SOLUTION_THRESHOLD_PCT:.4f}%")
print()
print("Now update fep_subtraction_2d_i11.py with these values:")
print(f"  In compute_frame_metrics(), change:")
print(f"    spot_threshold = {BEST_STD_MULTIPLIER}.0 * res_valid.std()")
print(f"    CRYSTAL_THRESHOLD  = {CRYSTAL_THRESHOLD_PCT/100:.6f}")
print(f"    SOLUTION_THRESHOLD = {SOLUTION_THRESHOLD_PCT/100:.6f}")
print()
print("Then come back and run Step 5b to verify.")

In [ ]:
# Step 5b — Verify classification with updated thresholds
# Run this AFTER updating fep_subtraction_2d_i11.py
# (Restart kernel and re-run Step 0 and Step 1 first)

import importlib, fep_subtraction_2d_i11
importlib.reload(fep_subtraction_2d_i11)
from fep_subtraction_2d_i11 import classify_dataset

csv_path = os.path.join(OUTPUT_DIR, "glycine_classification.csv")
all_metrics = classify_dataset(all_files, mask=mask, csv_output=csv_path)

# Compare against ground truth
import pandas as pd
df = pd.read_csv(csv_path)
df['true_label'] = df['collection_number'].astype(str).isin(KNOWN_CRYSTAL_NUMBERS)
df['true_label'] = df['true_label'].map({True: 'crystal', False: 'miss'})

crystal_auto = set(df[df['auto_class']=='crystal']['collection_number'].astype(str))
crystal_true = set(KNOWN_CRYSTAL_NUMBERS)

true_pos  = len(crystal_auto & crystal_true)
false_pos = len(crystal_auto - crystal_true)
false_neg = len(crystal_true - crystal_auto)

print(f"\n=== Classification vs ground truth ===")
print(f"  True positives  (correctly found crystal) : {true_pos} / {len(crystal_true)}")
print(f"  False positives (miss called crystal)     : {false_pos}")
print(f"  False negatives (crystal called miss)     : {false_neg}")
print()
if false_pos == 0 and false_neg == 0:
    print("✅ Perfect classification on glycine data!")
elif false_pos == 0:
    print(f"✅ No false positives (background pool is clean)")
    print(f"⚠️  {false_neg} crystal frames missed — threshold may be slightly too high")
else:
    print(f"⚠️  {false_pos} false positives — lower CRYSTAL_THRESHOLD slightly")
    
print(f"\nMissed crystal files:")
for n in sorted(crystal_true - crystal_auto):
    row = df[df['collection_number'].astype(str)==n]
    if len(row):
        print(f"  {n}  score={row.iloc[0]['crystal_score']:.4f}")

---
## Step 6 — Run full pipeline on glycine data

Once classification looks correct in Step 5, run the full subtraction pipeline.

In [ ]:
from fep_subtraction_2d_i11 import process_dataset

results = process_dataset(
    filepaths     = all_files,
    output_dir    = OUTPUT_DIR,
    scale_factor  = None,       # 1.0 (no count_time in metadata for this beamtime)
    clip_negative = True,
    mask          = mask,
    process_uncertain = False,
)

print(f"\nCorrected : {len(results['processed'])} files")
print(f"Skipped   : {len(results['skipped'])}")
print(f"Uncertain : {len(results['uncertain'])}")

# Check against ground truth
processed_numbers = set()
for p in results['processed']:
    stem = os.path.splitext(os.path.basename(str(p)))[0]
    for part in reversed(stem.replace('-','_').split('_')):
        if part.isdigit():
            processed_numbers.add(part)
            break

found    = processed_numbers & set(KNOWN_CRYSTAL_NUMBERS)
missed   = set(KNOWN_CRYSTAL_NUMBERS) - processed_numbers
extra    = processed_numbers - set(KNOWN_CRYSTAL_NUMBERS)
print(f"\nOf 50 known crystal files:")
print(f"  Correctly processed : {len(found)}")
print(f"  Missed              : {len(missed)}")
print(f"  Extra (false +ve)   : {len(extra)}")

---
## Step 7 — Visual check on a corrected glycine frame

Feeds one corrected frame through pyFAI and compares to the raw.
The FEP ring at ~5.6–6° 2θ should be significantly reduced.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

try:
    import pyFAI
    
    if len(results['processed']) == 0:
        print("No corrected files to check. Run Step 6 first.")
    else:
        from fep_subtraction_2d_i11 import _load_pixium_frame
        
        ai = pyFAI.load(PONI_PATH)
        npt = 3000
        
        # Pick first corrected file and its raw counterpart
        corr_fp = str(results['processed'][0])
        # Find matching raw file
        corr_stem = os.path.splitext(os.path.basename(corr_fp))[0]
        corr_num  = ""
        for part in reversed(corr_stem.replace('-','_').split('_')):
            if part.isdigit():
                corr_num = part; break
        raw_fp = next((f for f in all_files if corr_num in f), all_files[0])
        
        raw_frame  = _load_pixium_frame(raw_fp)
        corr_frame = _load_pixium_frame(corr_fp)
        if raw_frame.ndim  == 3: raw_frame  = raw_frame[0]
        if corr_frame.ndim == 3: corr_frame = corr_frame[0]
        
        mask_int = mask.astype(np.int32)
        r_raw  = ai.integrate1d(raw_frame.astype(np.float32),  npt, mask=mask_int, unit="2th_deg")
        r_corr = ai.integrate1d(corr_frame.astype(np.float32), npt, mask=mask_int, unit="2th_deg")
        
        tth_raw,  i_raw  = np.array(r_raw[0]),  np.array(r_raw[1])
        tth_corr, i_corr = np.array(r_corr[0]), np.array(r_corr[1])
        
        fig, axes = plt.subplots(2, 1, figsize=(12, 10))
        
        # Full pattern
        axes[0].plot(tth_raw,  i_raw,  lw=1.0, color='coral', alpha=0.8, label='Raw (with FEP)')
        axes[0].plot(tth_corr, i_corr, lw=1.2, color='teal',  label='2D FEP corrected')
        axes[0].set_xlabel("2θ (°)"); axes[0].set_ylabel("Intensity")
        axes[0].set_title("Full pattern: raw vs 2D FEP corrected")
        axes[0].legend(); axes[0].grid(alpha=0.3)
        
        # Low-angle zoom — FEP peak region
        lo, hi = 4.5, 9.0
        for ax, tth_r, i_r, tth_c, i_c in [(axes[1], tth_raw, i_raw, tth_corr, i_corr)]:
            m_r = (tth_r>=lo)&(tth_r<=hi)
            m_c = (tth_c>=lo)&(tth_c<=hi)
            ax.plot(tth_r[m_r], i_r[m_r], lw=1.0, color='coral', alpha=0.8, label='Raw')
            ax.plot(tth_c[m_c], i_c[m_c], lw=1.5, color='teal',  label='Corrected')
            ax.axvspan(5.4, 6.2, alpha=0.08, color='gray', label='FEP peak region')
            ax.set_xlabel("2θ (°)"); ax.set_ylabel("Intensity")
            ax.set_title(f"Low-angle zoom ({lo}–{hi}°) — FEP peak region")
            ax.legend(); ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "glycine_1D_comparison.png"), dpi=150)
        plt.show()
        
        # Reduction at FEP peak
        fep_m_r = (tth_raw>=5.4)&(tth_raw<=6.2)
        fep_m_c = (tth_corr>=5.4)&(tth_corr<=6.2)
        fep_raw  = i_raw[fep_m_r].max()
        fep_corr = i_corr[fep_m_c].max()
        print(f"FEP peak max (raw)      : {fep_raw:.0f}")
        print(f"FEP peak max (corrected): {fep_corr:.0f}")
        print(f"Reduction               : {100*(fep_raw-fep_corr)/fep_raw:.1f}%")

except ImportError:
    print("pyFAI not available — skip Step 7.")

---
## Step 8 — Record results

In [ ]:
print("=" * 60)
print("  GLYCINE CALIBRATION TRIAL — SUMMARY")
print("=" * 60)
print(f"  Dataset          : Run7_GLY_0.5VF_X2")
print(f"  Total files      : {len(all_files)}")
print(f"  Known crystal    : {len(KNOWN_CRYSTAL_NUMBERS)}")
if 'results' in dir():
    print(f"  Auto-classified  : {len(results['processed'])} crystal")
    print(f"  Uncertain        : {len(results['uncertain'])}")
print()
print("  Fill in manually:")
print("  FEP ring r confirmed at  : _____ px (from Step 3)")
print("  Best threshold multiplier: _____ std")
print("  CRYSTAL_THRESHOLD used   : _____")
print("  True positives           : _____ / 50")
print("  False positives          : _____")
print("  FEP peak reduction       : _____% (from Step 7)")
print("  Ready to send to engineer: YES / NO")